# Biomedical Knowledge Graph: End-to-End Example

**Complete workflow from PubMed abstracts to knowledge graph:**
1. Fetch PubMed abstracts
2. Extract entities with biomedical NER
3. Link entities to UMLS ontology
4. Build knowledge graph
5. Query the knowledge graph

**Expected runtime:** ~3-5 minutes for 10 abstracts


In [1]:
# Setup
import os
import pandas as pd
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

# Import pipeline components
from biomed_kg_agent.data_sources.ncbi import pubmed
from biomed_kg_agent.core.pipeline import process_documents
from biomed_kg_agent.core.connectors import pubmed_to_internal
from biomed_kg_agent.orchestrators import build_knowledge_graph

load_dotenv()

print("Biomedical KG pipeline loaded")


Biomedical KG pipeline loaded


## Step 1: Fetch PubMed Abstracts

Retrieve recent papers on a topic of interest.


In [2]:
# Define your research query
query = "breast cancer AND immunotherapy AND 2024[PDAT]"
max_papers = 10

print(f"Searching PubMed: '{query}'")
print(f"Retrieving up to {max_papers} abstracts...\n")

# Fetch from PubMed
pmids = pubmed.get_pubmed_pmids(query, limit=max_papers)
pubmed_docs = pubmed.fetch_abstracts(pmids)

print(f"Retrieved {len(pubmed_docs)} abstracts")
print(f"\nExample abstract:")
print(f"   PMID: {pubmed_docs[0].pmid}")
print(f"   Title: {pubmed_docs[0].title[:100]}...")


Searching PubMed: 'breast cancer AND immunotherapy AND 2024[PDAT]'
Retrieving up to 10 abstracts...

Retrieved 10 abstracts

Example abstract:
   PMID: 40880982
   Title: Longitudinal Monitoring of T cell Dynamics in Metastatic Breast Cancer via a Remote Diagnostic Impla...


## Step 2: Extract Biomedical Entities

Apply scispaCy NER models to extract diseases, chemicals, genes, etc.


In [3]:
# Create demo directory for output files
demo_dir = "data/demo"
os.makedirs(demo_dir, exist_ok=True)
nlp_db = os.path.join(demo_dir, "nlp_results.db")
print(f"Using directory: {demo_dir}\n")

# Convert to internal format
internal_docs = [pubmed_to_internal(doc) for doc in pubmed_docs]

print("Running biomedical NER + UMLS linking...")
print("   (This may take a few minutes for model loading + processing)\n")

# Process documents with database output (uses all 3 scispaCy models)
# The two-pass processor requires a database for caching UMLS mappings
processed_docs = process_documents(
    internal_docs, 
    config_path=None,  # Use default config
    output_db=nlp_db   # Required for UMLS caching
)

# Calculate statistics
total_entities = sum(len(doc.entities) for doc in processed_docs)
umls_linked = sum(
    sum(1 for e in doc.entities if e.umls_cui) for doc in processed_docs
)
umls_rate = (umls_linked / total_entities * 100) if total_entities > 0 else 0

print(f"\nEntity extraction complete!")
print(f"   {total_entities} entities extracted")
print(f"   {umls_linked} linked to UMLS ({umls_rate:.1f}%)")
print(f"   {len(processed_docs)} documents processed")
print(f"   Results saved to: {nlp_db}")


Using directory: data/demo

Running biomedical NER + UMLS linking...
   (This may take a few minutes for model loading + processing)


Entity extraction complete!
   250 entities extracted
   211 linked to UMLS (84.4%)
   10 documents processed
   Results saved to: data/demo/nlp_results.db


### Quick Look at Extracted Entities


In [4]:
# Show entities from first document
sample_doc = processed_docs[0]
sample_entities = sample_doc.entities[:10]  # First 10 entities

print(f"Sample entities from PMID {sample_doc.id}:\n")
for i, entity in enumerate(sample_entities, 1):
    umls_info = f"-> {entity.umls_preferred_name} ({entity.umls_cui})" if entity.umls_cui else "(no UMLS)"
    print(f"   {i}. '{entity.text}' [{entity.entity_type}] {umls_info}")

if len(sample_doc.entities) > 10:
    print(f"\n   ... and {len(sample_doc.entities) - 10} more entities")


Sample entities from PMID 40880982:

   1. 'tumor' [disease] -> Neoplasms (C0027651)
   2. 'breast cancer' [disease] -> Malignant neoplasm of breast (C0006142)
   3. 'cancer' [disease] -> Malignant Neoplasms (C0006826)
   4. 'tumors' [disease] -> Neoplasms (C0027651)
   5. 'tumor' [disease] -> Neoplasms (C0027651)
   6. '4T1 tumors' [disease] -> Neoplasms (C0027651)
   7. 'BC' [chemical] -> BC Original Formula (C3267881)
   8. 'TME' [chemical] (no UMLS)
   9. 'anticancer' [chemical] -> Antineoplastic Agents (C0003392)
   10. 'TME' [chemical] (no UMLS)

   ... and 18 more entities


## Step 3: Inspect Processed Documents

The NLP results are already saved in the database. Let's verify what we have.


In [5]:
# Verify NLP database was created
import sqlite3

conn = sqlite3.connect(nlp_db)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
conn.close()

print(f"NLP database created with {len(tables)} tables:")
print(f"   {', '.join(tables['name'].tolist())}")
print(f"\n   Note: Both NLP and KG databases share the same schema (from models.py)")
print(f"         but populate different tables:")
print(f"         - NLP DB: processed_documents, extracted_entities (NER results)")
print(f"         - KG DB: entity, mention, cooccurrence (graph structure)")
print(f"\n   Ready for knowledge graph construction!")


NLP database created with 6 tables:
   pubmeddocument, extracted_entities, processed_documents, entity, mention, cooccurrence

   Note: Both NLP and KG databases share the same schema (from models.py)
         but populate different tables:
         - NLP DB: processed_documents, extracted_entities (NER results)
         - KG DB: entity, mention, cooccurrence (graph structure)

   Ready for knowledge graph construction!


## Step 4: Build Knowledge Graph

Extract co-occurrence relationships between entities (sentence-level).


In [6]:
# Build KG from processed documents
kg_db = os.path.join(demo_dir, "kg.db")

print("Building knowledge graph (co-occurrence extraction)...\n")

kg_result = build_knowledge_graph(nlp_db, kg_db)

print(f"\nKnowledge graph built!")
print(f"   Entities: {kg_result['entities']}")
print(f"   Relationships: {kg_result['cooccurrences']}")
print(f"   Mentions: {kg_result['mentions']}")
print(f"   Saved to: {kg_db}")


Building knowledge graph (co-occurrence extraction)...


Knowledge graph built!
   Entities: 100
   Relationships: 320
   Mentions: 248
   Saved to: data/demo/kg.db


## Step 5: Query the Knowledge Graph

The KG is stored in SQLite. Let's explore it!


In [7]:
# Connect to KG database
conn = sqlite3.connect(kg_db)

# List tables
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)
print("Knowledge Graph Database:")
print(f"Tables: {', '.join(tables['name'].tolist())}")
print(f"\nThe KG database contains the final graph structure:")
print(f"   - entity: Unique biomedical entities (UMLS-normalized)")
print(f"   - mention: Entity occurrences in documents")
print(f"   - cooccurrence: Relationships between entities")
print()


Knowledge Graph Database:
Tables: pubmeddocument, extracted_entities, processed_documents, entity, mention, cooccurrence

The KG database contains the final graph structure:
   - entity: Unique biomedical entities (UMLS-normalized)
   - mention: Entity occurrences in documents
   - cooccurrence: Relationships between entities



### Query 1: Most Frequently Mentioned Entities


In [8]:
query = """
SELECT 
    e.name,
    e.umls_preferred_name AS umls_name,
    e.entity_type AS type,
    e.umls_cui,
    COUNT(DISTINCT m.doc_id) as doc_count,
    COUNT(*) as mention_count
FROM entity e
JOIN mention m ON e.id = m.entity_id
GROUP BY e.id
ORDER BY doc_count DESC, mention_count DESC
LIMIT 15
"""

top_entities = pd.read_sql(query, conn)
print("Top 15 Most Mentioned Entities:\n")
print(top_entities.to_string(index=False))


Top 15 Most Mentioned Entities:

                         name                        umls_name      type umls_cui  doc_count  mention_count
                       cancer              Malignant Neoplasms   disease C0006826          7             81
                       tumors                        Neoplasms   disease C0027651          6             36
                breast cancer     Malignant neoplasm of breast   disease C0006142          5             36
                      patient                         Patients  organism C0030705          5             18
                        cells                             None cell_type     None          4             15
                         cell                            Cells cell_type C0007634          3             12
                         mice                       House mice  organism C0025914          3              9
                   stem cells                       Stem cells cell_type C0038250          2           

### Query 2: Entity Type Distribution


In [9]:
query = """
SELECT 
    entity_type,
    COUNT(DISTINCT id) as unique_entities,
    COUNT(DISTINCT CASE WHEN umls_cui IS NOT NULL THEN id END) as umls_linked,
    ROUND(COUNT(DISTINCT CASE WHEN umls_cui IS NOT NULL THEN id END) * 100.0 / COUNT(DISTINCT id), 1) as umls_rate
FROM entity
GROUP BY entity_type
ORDER BY unique_entities DESC
"""

entity_types = pd.read_sql(query, conn)
print("Entity Type Distribution:\n")
print(entity_types.to_string(index=False))


Entity Type Distribution:

       entity_type  unique_entities  umls_linked  umls_rate
          chemical               29           23       79.3
           disease               21           15       71.4
         cell_type               16           10       62.5
              gene                8            6       75.0
  sequence_feature                6            3       50.0
          organism                6            6      100.0
           anatomy                5            5      100.0
         substance                4            4      100.0
biological_process                3            3      100.0
         pathology                1            1      100.0
cellular_component                1            1      100.0


### Query 3: Strongest Relationships (Multi-Document Evidence)


In [10]:
query = """
SELECT 
    e1.name as entity_1,
    e1.entity_type as type_1,
    e2.name as entity_2,
    e2.entity_type as type_2,
    r.docs_count as documents,
    r.sent_count as sentences
FROM cooccurrence r
JOIN entity e1 ON r.entity_a_id = e1.id
JOIN entity e2 ON r.entity_b_id = e2.id
WHERE r.docs_count >= 2  -- At least 2 independent papers
ORDER BY r.docs_count DESC, r.sent_count DESC
LIMIT 20
"""

relationships = pd.read_sql(query, conn)

if len(relationships) > 0:
    print("Top 20 Relationships (Multi-Document Evidence):\n")
    print(relationships.to_string(index=False))
else:
    print("No multi-document relationships found in this small corpus.")
    print("   (Expected for <20 abstracts - try single-document relationships below)")


Top 20 Relationships (Multi-Document Evidence):

                     entity_1    type_1                      entity_2    type_2  documents  sentences
                   stem cells cell_type             cancer stem cells cell_type          2          7
                       cancer   disease                    stem cells cell_type          2          6
                       cancer   disease             cancer stem cells cell_type          2          6
                       cancer   disease                        tumors   disease          2          3
                       tumors   disease                    stem cells cell_type          2          3
                       tumors   disease             cancer stem cells cell_type          2          3
                breast cancer   disease                       patient  organism          2          2
                        cells cell_type                    tnbc cells cell_type          2          2
                        cells cel

### Query 4: All Relationships (Including Single-Document)


In [11]:
query = """
SELECT 
    e1.name as entity_1,
    e1.entity_type as type_1,
    e2.name as entity_2,
    e2.entity_type as type_2,
    r.sent_count as co_occurrences
FROM cooccurrence r
JOIN entity e1 ON r.entity_a_id = e1.id
JOIN entity e2 ON r.entity_b_id = e2.id
ORDER BY r.sent_count DESC
LIMIT 20
"""

all_relationships = pd.read_sql(query, conn)
print("Top 20 Entity Relationships (All):\n")
print(all_relationships.to_string(index=False))


Top 20 Entity Relationships (All):

   entity_1    type_1                      entity_2    type_2  co_occurrences
 stem cells cell_type             cancer stem cells cell_type               7
     cancer   disease             cancer stem cells cell_type               6
     cancer   disease                    stem cells cell_type               6
    disease   disease                        t cell cell_type               4
       cscs   disease                         pd-l1  chemical               4
       lung   anatomy                        t cell cell_type               3
     tumors   disease             cancer stem cells cell_type               3
doxorubicin  chemical                         pd-l1  chemical               3
     tumors   disease                    stem cells cell_type               3
doxorubicin  chemical                          cscs   disease               3
     cancer   disease                        tumors   disease               3
doxorubicin  chemical       

In [12]:
conn.close()

In [13]:
print(f"\nDemo complete!")
print(f"\nYour files are saved in: {demo_dir}/")
print(f"   - NLP results: {nlp_db}")
print(f"   - Knowledge graph: {kg_db}")
print(f"\nTo clean up demo files:")
print(f"   rm -rf {demo_dir}")



Demo complete!

Your files are saved in: data/demo/
   - NLP results: data/demo/nlp_results.db
   - Knowledge graph: data/demo/kg.db

To clean up demo files:
   rm -rf data/demo
